# 小作业：模型能补全“比利时位于欧洲”吗？

姓名：____　学号：____

**核心案例：训练集有“比利时位于西欧”“西欧位于欧洲”，但没有“比利时位于欧洲”。**
我们先看模型的预测，再训练一次，检查它能否补全这条事实。

按顺序运行代码，填写3道短答题。无需写代码或修改参数。


In [ ]:
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt
from kge_homework import load_data, run_assignment, show_clues, belgium_case

train, valid, test, _ = load_data()
print(f"实体：{train.num_entities}；关系：{train.num_relations}")
print(f"训练三元组：{train.num_triples}；验证：{valid.num_triples}；测试：{test.num_triples}")

## 先看核心案例
下面直接核对数据文件：两条线索在训练集，完整答案只在测试集。
训练时不向模型提供这条测试答案。


In [ ]:
display(belgium_case())

## 1. 训练前后各答一次题
代码先记录随机初始化模型的预测，再训练同一个模型100轮，最后重新预测。运行期间请等待，不要重复点击。

In [ ]:
result = run_assignment()
print("训练前")
display(result["before"]["predictions"])
print("训练后")
display(result["after"]["predictions"])

## 2. 看看有没有进步
表中正确率针对全部24道测试题，每题只从5个大地区中选答案。不是标准链接预测指标。

In [ ]:
display(pd.DataFrame([result["before"]["summary"], result["after"]["summary"]]))
print(f"训练耗时：{result['training_seconds']} 秒")
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(result["losses"])+1), result["losses"])
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.tight_layout()
plt.show()

## 3. 重点分析：比利时位于哪里？（必做）
代码固定展示比利时训练前后的预测，以及两条训练集线索。
观察：正确答案“欧洲”的排名变了吗？训练后第一名与线索支持的答案一致吗？

**补全事实与解释模型要分开：答对了，不代表模型内部一定按这两步推理。**


In [ ]:
display(belgium_case())
belgium_rows = []
for stage, label in [('before', '训练前'), ('after', '训练后')]:
    row = result[stage]['predictions'].query("实体 == 'belgium'").copy()
    row.insert(0, '阶段', label)
    belgium_rows.append(row)
belgium_comparison = pd.concat(belgium_rows, ignore_index=True)
display(belgium_comparison)


### 补充观察：有线索，模型也可能答错
如有错题，程序自动展示一道及其训练集线索，帮助回答第3题。没有错题则跳过，无需额外训练或回答。


In [ ]:
errors = result['after']['errors']
if not errors.empty:
    selected = errors.iloc[0]
    print(f"错题：{selected['country']}；模型第一名：{selected['top1']}；正确答案：{selected['answer']}")
    display(show_clues(selected['country']))
else:
    print('本次24题全部答对；这不能保证模型在其他题上也全部答对。')


## 4. 回答问题
每题2—4句话，引用你自己的运行结果。双击下面的文字框填写。

**1.** 训练前后分别答对多少题？说明训练带来了什么变化。

答：训练前答对2/24题，正确率为8.33%；训练后答对23/24题，正确率为95.83%。训练后的正确率提高，答对题数变化为+21题，说明学习训练集事实改变了模型对候选地区的排序。这个结果只反映本次24道五选一测试题的表现，不能直接等同于标准链接预测指标或对其他数据的表现。

**2. 比利时案例：** “欧洲”在训练前后分别排第几？根据两条训练事实，你能得出什么答案？为什么这属于补全训练集中没有直接出现的事实？

答：正确答案“欧洲”在训练前排第4，训练后排第1。训练集中的“比利时位于西欧”和“西欧位于欧洲”在地理包含关系下支持“比利时位于欧洲”。这条完整三元组没有直接出现在训练集中，而出现在测试集中，因此预测它属于补全未直接提供的事实，训练时没有把该测试答案加入训练。


**3. 比利时案例：** 训练后模型的第一名与地理线索支持的答案一致吗？即使答对，能否证明模型内部按这两步推理？有线索是否保证一定答对？

答：训练后模型的第一名是europe，与线索支持的欧洲一致。即使模型答对，也不能证明它内部显式执行了“比利时→西欧→欧洲”的两步推理；TransE通过实体向量与关系向量的距离打分，预测结果本身不能确定它利用了哪条路径。有线索也不保证一定答对，模型的向量表示、训练目标和有限训练可能使其他候选地区得分更高；本次程序展示的错题是paraguay，模型第一名为africa，参考答案为americas，也说明这次学习没有保证所有测试题正确。


保存为 `学号_姓名.ipynb`，保留运行结果，写完回答，只提交这个文件。